In [ ]:
%pip install pandas
%pip install torch --index-url https://download.pytorch.org/whl/cu118
%pip install transformers
%pip install scikit-learn

In [ ]:
# Importing the libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Load the local IMDB dataset
df = pd.read_csv('IMDB Dataset.csv')
df.head(5)

In [ ]:
# Create train and test splits (80-20 split)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        review = str(self.data.iloc[idx]['review'])
        sentiment = self.data.iloc[idx]['sentiment']
        
        # Convert sentiment to numeric
        label = 1 if sentiment == 'positive' else 0
        
        # Tokenize the text
        encoding = self.tokenizer(
            review,
            #add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Initialize the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Create dataset objects
train_dataset = IMDBDataset(train_df, tokenizer)
test_dataset = IMDBDataset(test_df, tokenizer)

# Create data loaders
batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Initialize the BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2)

# Freeze the base model parameters
for param in model.distilbert.parameters():
    param.requires_grad = False

# Only the classification head parameters will be updated during training
for param in model.classifier.parameters():
    param.requires_grad = True

# Check if CUDA is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training settings
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=2e-5)
epochs = 1

In [ ]:
# Training loop
def train():
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            total_loss += loss.item()
            
            loss.backward()
            optimizer.step()
            
            # Print progress every 100 batches
            if (batch_idx + 1) % 100 == 0:
                print(f'Epoch {epoch + 1}/{epochs}, Batch {batch_idx + 1}/{len(train_loader)}, Loss: {loss.item():.4f}')
            
        avg_loss = total_loss / len(train_loader)
        print(f'Epoch {epoch + 1}/{epochs}, Average Loss: {avg_loss:.4f}')

In [ ]:
# Evaluate the model
def evaluate():
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            
            predictions = torch.argmax(outputs.logits, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total
    print(f'Test Accuracy: {accuracy:.4f}')

In [ ]:
# Train the model
train()

In [ ]:
# Evaluate the model
evaluate()

In [ ]:
def predict_sentiment(text):
    model.eval()
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        prediction = torch.argmax(outputs.logits, dim=1)
        
    return 'positive' if prediction.item() == 1 else 'negative'

# Test the model with a sample text
sample_text = "This movie was really great! I enjoyed every minute of it."
print(f"Sentiment: {predict_sentiment(sample_text)}")